In [8]:
import pandas as pd
import json
import numpy as np
import urllib.request
import re, unicodedata

In [9]:
import os

BASE = "https://raw.githubusercontent.com/Nick-Anthony/velora/main/"
ACCOUNTS_FILE = "accounts.csv"
EVENTS_FILE   = "engagement_signals.json"

TIER_WEIGHTS = {"Strategic": 0.89, "Enterprise": 0.85, "Mid-Market": 0.58, "SMB": 0.22}
TIER_FALLBACK = 0.4


def get_data_files():
    """Fetch the exports from the repo, caching to disk. If the fetch fails
    (GitHub rate-limits its raw endpoint), fall back to a manual upload."""
    if os.path.exists(ACCOUNTS_FILE) and os.path.exists(EVENTS_FILE):
        print("Using files already in the working directory.")
        return

    try:
        req = lambda p: urllib.request.Request(
            BASE + p, headers={"User-Agent": "Mozilla/5.0"})
        for fname in (ACCOUNTS_FILE, EVENTS_FILE):
            with urllib.request.urlopen(req(fname)) as r, open(fname, "wb") as out:
                out.write(r.read())
        print(f"Downloaded {ACCOUNTS_FILE} and {EVENTS_FILE} from the repo.")
        return
    except Exception as e:
        print(f"Could not download from the repo ({e}).")

    print(f"Please upload {ACCOUNTS_FILE} and {EVENTS_FILE} "
          f"(both are in the GitHub repo).")
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        raise SystemExit(
            f"Not running in Colab. Place {ACCOUNTS_FILE} and {EVENTS_FILE} "
            f"next to this notebook and re-run.")

    missing = [f for f in (ACCOUNTS_FILE, EVENTS_FILE) if not os.path.exists(f)]
    if missing:
        raise SystemExit(f"Still missing: {missing}")


get_data_files()

accounts = pd.read_csv(ACCOUNTS_FILE, dtype="str")
with open(EVENTS_FILE) as f:
    events = pd.DataFrame(json.load(f))
events["event_date"] = pd.to_datetime(events["event_date"], errors="coerce")

# Reference date: the latest date the export can vouch for. The engagement stream
# is machine-generated, so its last timestamp is the most reliable statement of
# when the export was taken. The 98th-percentile contact date is a fallback that
# ignores stray far-future typos. Anchoring here rather than on the wall clock
# means the same export always produces the same ranking.
_contact = pd.to_datetime(accounts["last_contact_date"], errors="coerce")
REF_DATE = max(events["event_date"].max(), _contact.quantile(0.98))
print("Reference date derived from data:", REF_DATE.date())

Using files already in the working directory.
Reference date derived from data: 2026-07-28


| tier | n | median ARR (USD) | median ARR percentile |
|---|---|---|---|
| Enterprise | 42 | 82,225 | 0.89 |
| Strategic | 37 | 75,200 | 0.85 |
| Mid-Market | 93 | 33,100 | 0.58 |
| SMB | 120 | 9,750 | 0.22 |

Tier weights are set to the median ARR percentile of accounts in each tier, so that tier acts as a revenue proxy for the 40 accounts where ARR is unusable.

Eight accounts have no usable tier (5 missing, 3 labelled `TBD`). These receive a
fallback weight of 0.4 rather than 0 — an unlabelled account is unknown-sized, not
small, so it should rank mid-pack rather than be pushed to the bottom. The three
`TBD` accounts sit at a median ARR percentile of 0.44, which supports 0.4 as a
reasonable neutral value.

In [10]:
FLAG_RULES = {
    "arr_missing":  "ARR missing — imputed median",
    "arr_invalid":  "ARR value was not a valid number",
    "date_missing": "Last contact date missing — neglect set to neutral",
    "date_future":  "Last contact date was in the future — treated as missing",
    "tier_missing": "Account tier missing or TBD — default weight applied",
    "no_events":    "No engagement signals found",
    "was_merged":   "Merged from duplicate CRM records",
}

In [11]:
print(len(accounts))
print(accounts.isna().sum())

300
account_name          0
industry             60
arr                  38
last_contact_date    23
account_tier          5
website               0
region                0
owner                 0
dtype: int64


Coerce four working columns

In [12]:
accounts["arr_num"] = pd.to_numeric(accounts["arr"], errors="coerce")
accounts["arr_invalid"] = (
    (accounts["arr_num"].isna() & accounts["arr"].notna())   # unparseable, e.g. "banana"
    | (accounts["arr_num"] < 0)                              # parses but implausible
)
accounts.loc[accounts["arr_num"] < 0, "arr_num"] = pd.NA
accounts["arr_num"] = pd.to_numeric(accounts["arr_num"])

accounts["contact_date"] = pd.to_datetime(accounts["last_contact_date"], errors="coerce")
accounts["date_future"] = accounts["contact_date"] > REF_DATE
accounts.loc[accounts["date_future"], "contact_date"] = pd.NaT

accounts["tier_weight"] = accounts["account_tier"].map(TIER_WEIGHTS)

accounts["domain"] = (
    accounts["website"].fillna("").str.strip()
    .str.replace(r"^https?://", "", regex=True)
    .str.replace(r"^www\.", "", regex=True)
    .str.split("/").str[0]
    .str.lower()
)

verify


In [13]:
print("arr unusable:", accounts["arr_num"].isna().sum())        # 40
print("date unusable:", accounts["contact_date"].isna().sum())  # 27
print("tier unusable:", accounts["tier_weight"].isna().sum())   # 8
print("arr invalid:", accounts["arr_invalid"].sum())   # 2
print("date future:", accounts["date_future"].sum())   # 4

arr unusable: 40
date unusable: 27
tier unusable: 8
arr invalid: 2
date future: 4


Merge duplicate companies


In [14]:
valid_domain = (
    accounts["domain"].str.match(r"^[a-z0-9-]+(\.[a-z0-9-]+)+$", na=False)
    & ~accounts["domain"].str.contains("@", na=False)
)

print("valid:", valid_domain.sum(), "invalid:", (~valid_domain).sum())
print(accounts.loc[~valid_domain, ["account_name", "website"]])

valid: 297 invalid: 3
              account_name              website
194  Lutheran World Relief    hello@example.net
200                   BRAC  contact@example.org
276                    NPR     team@example.com


In [15]:
groupable = accounts[valid_domain]
passthrough = accounts[~valid_domain].copy()

In [16]:
merged = groupable.groupby("domain", as_index=False).agg(
    account_name=("account_name", lambda s: max(s, key=len)),
    arr_num=("arr_num", "max"),
    contact_date=("contact_date", "max"),
    tier_weight=("tier_weight", "max"),
    account_tier=("account_tier", lambda s: s.dropna().iloc[0] if s.notna().any() else None),
    industry=("industry", lambda s: s.dropna().iloc[0] if s.notna().any() else None),
    region=("region", "first"),
    owner=("owner", lambda s: " / ".join(sorted(s.dropna().unique()))),
    source_names=("account_name", lambda s: " | ".join(sorted(s))),
    source_arr=("arr_num", lambda s: " | ".join(f"{v:,.0f}" for v in s.dropna())),
    row_count=("account_name", "size"),
    arr_invalid=("arr_invalid", "any"),
    date_future=("date_future", "any"),
)

print(len(groupable), "->", len(merged))

297 -> 283


### Merging duplicate accounts

The export contains 14 organisations recorded twice under different names —
`AFSC` and `American Friends Service Committee`, `WWF` and `World Wildlife Fund`,
`IRC` and `International Rescue Committee`. Name similarity cannot detect these:
an initialism shares almost no characters with the full name it abbreviates.
The website domain is the only reliable link between them, so records are merged
on normalised domain rather than on name.

Each column is collapsed on its own rule:

| column | rule | reasoning |
|---|---|---|
| `account_name` | longest | full names are more recognisable to an SDR than initialisms |
| `arr_num` | max | the pairs disagree; see below |
| `contact_date` | most recent | both contacts happened, and the latest one is what neglect should measure |
| `tier_weight` | max | mapped to numbers before merging, so `max` compares 0.22 against 0.89 rather than sorting strings alphabetically |
| `account_tier` | first non-null | preserves a label even when one record is blank |
| `owner` | all distinct, joined | two reps on one account is itself a finding worth surfacing |
| `source_names`, `source_arr` | both values kept | audit trail — the merge is inspectable rather than lossy |

**On conflicting ARR.** Every merged pair carries two different revenue figures
(AFSC is recorded as both 77,100 and 92,450). The export has no modification
timestamp, so there is no principled way to identify which value is current.
The merge takes the maximum on the assumption that the fuller record is the more
complete one, retains both original values in `source_arr`, and flags the account
as merged so the figure can be verified before use. This is a stated assumption,
not a resolution.

**Non-goal.** No attempt is made to detect duplicates that share neither a domain
nor a near-identical name. Reliable entity resolution needs either a firmographic
data source or human review, both outside MVP scope.

In [17]:
passthrough["source_names"] = passthrough["account_name"]
passthrough["source_arr"] = passthrough["arr_num"].apply(
    lambda v: "" if pd.isna(v) else f"{v:,.0f}")
passthrough["row_count"] = 1

accounts_merged = pd.concat([merged, passthrough[merged.columns]], ignore_index=True)
print(len(accounts_merged))   # 286

286


In [18]:
print(accounts_merged[accounts_merged["row_count"] > 1][
    ["account_name", "source_names", "source_arr"]].to_string(index=False))
print(accounts_merged[["arr_invalid", "date_future"]].sum())   # 2 and 4

                                             account_name                                                      source_names       source_arr
                       American Friends Service Committee                         AFSC | American Friends Service Committee  77,100 | 92,450
American Society for the Prevention of Cruelty to Animals ASPCA | American Society for the Prevention of Cruelty to Animals  38,700 | 43,900
                                 Catholic Relief Services                                    CRS | Catholic Relief Services  28,750 | 38,950
                             Médecins Sans Frontières USA            Doctors Without Borders | Médecins Sans Frontières USA  90,600 | 78,050
                               Environmental Defense Fund                                  EDF | Environmental Defense Fund 120,000 | 95,150
                       Habitat for Humanity International         Habitat for Humanity | Habitat for Humanity International  20,950 | 25,150
             

Set confidence flags

In [19]:
accounts_merged["arr_missing"]  = accounts_merged["arr_num"].isna()
accounts_merged["date_missing"] = accounts_merged["contact_date"].isna()
accounts_merged["tier_missing"] = accounts_merged["tier_weight"].isna()
accounts_merged["was_merged"]   = accounts_merged["row_count"] > 1

print(accounts_merged[["arr_missing", "date_missing", "tier_missing", "was_merged"]].sum())

arr_missing     40
date_missing    27
tier_missing     8
was_merged      14
dtype: int64


In [20]:
ARR_MEDIAN = accounts_merged["arr_num"].median()
accounts_merged["arr_num"] = accounts_merged["arr_num"].fillna(ARR_MEDIAN)
accounts_merged["tier_weight"] = accounts_merged["tier_weight"].fillna(TIER_FALLBACK)

JSON


In [21]:


ABBREV = {
    r"\bfdn\b": "foundation",   r"\binst\b": "institute",
    r"\bintl\b": "international", r"\bdev\b": "development",
    r"\bcomm\b": "community",   r"\bamer\b": "america",
    r"\bww\b": "worldwide",
}
SUFFIX = r"\b(incorporated|inc|llc|ltd|corp|corporation|co|usa|us|the)\b"

def norm(s):
    if pd.isna(s):
        return ""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    s = s.lower()
    s = re.sub(r"[.,&']", " ", s)          # punctuation to spaces, so \b still works
    for pat, rep in ABBREV.items():
        s = re.sub(pat, rep, s)
    s = re.sub(SUFFIX, " ", s)
    s = re.sub(r"[^a-z0-9]", "", s)        # collapse everything remaining
    return s

In [22]:
events["key"] = events["account_name"].map(norm)

alias = {}
for _, row in accounts_merged.iterrows():
    for name in row["source_names"].split(" | "):
        alias[norm(name)] = row["account_name"]

events["matched_account"] = events["key"].map(alias)
print("unmatched event rows:", events["matched_account"].isna().sum())

unmatched event rows: 0


In [23]:
print(events["matched_account"].nunique(), "distinct accounts have events")
print(len(accounts_merged) - events["matched_account"].nunique(), "accounts have none")

211 distinct accounts have events
75 accounts have none


In [24]:
EVENT_WEIGHTS = {"demo_request": 10, "webinar": 5, "content_download": 4,
                 "page_visit": 2, "email_open": 1}
HALF_LIFE_DAYS = 30

events["event_date"] = pd.to_datetime(events["event_date"], errors="coerce")

days_ago = (REF_DATE - events["event_date"]).dt.days
events["decay"] = 0.5 ** (days_ago / HALF_LIFE_DAYS)
events["weight"] = events["event_type"].map(EVENT_WEIGHTS)
events["damped"] = np.log1p(events["event_count"])

events["score"] = events["weight"] * events["damped"] * events["decay"]

In [25]:
intent_raw = events.groupby("matched_account")["score"].sum()
intent = intent_raw / intent_raw.max()

In [26]:
accounts_merged["intent"] = accounts_merged["account_name"].map(intent).fillna(0)
accounts_merged["no_events"] = accounts_merged["account_name"].map(intent).isna()

In [27]:
accounts_merged["fit"] = (
    0.6 * accounts_merged["arr_num"].rank(pct=True)
    + 0.4 * accounts_merged["tier_weight"]
)

days_since = (REF_DATE - accounts_merged["contact_date"]).dt.days
accounts_merged["neglect"] = (days_since / 180).clip(0, 1).fillna(0.5)

accounts_merged["priority"] = accounts_merged["fit"] * (
    0.7 * accounts_merged["intent"] + 0.3 * accounts_merged["neglect"]
)

In [28]:
ranked = accounts_merged.sort_values("priority", ascending=False)
print(ranked.head(20)[["account_name", "priority", "fit", "intent", "neglect",
                       "arr_num", "account_tier"]].to_string(index=False))

                                             account_name  priority      fit   intent  neglect  arr_num account_tier
                                         Alley Cat Allies  0.586703 0.850839 0.904131 0.188889  70450.0   Enterprise
                       American Friends Service Committee  0.580826 0.918238 0.475064 1.000000  92450.0    Strategic
                             Education Development Center  0.566892 0.938951 0.776786 0.200000 120000.0   Enterprise
                                            No Kid Hungry  0.524592 0.641049 1.000000 0.394444  26850.0   Enterprise
                                                     PATH  0.482704 0.869720 0.483348 0.722222  76250.0   Enterprise
                                            Christian Aid  0.464757 0.837203 0.364472 1.000000  64400.0   Enterprise
                       Save the Children Federation, Inc.  0.451965 0.889650 0.675751 0.116667  84600.0   Enterprise
                            Cold Spring Harbor Laboratory  0.439

In [29]:
CONF_WEIGHTS = {
    "arr_missing":  0.30,
    "no_events":    0.30,
    "tier_missing": 0.10,
    "date_missing": 0.10,
}

penalty = sum(accounts_merged[k] * w for k, w in CONF_WEIGHTS.items())
accounts_merged["confidence_score"] = 1 - penalty

accounts_merged["confidence"] = pd.cut(
    accounts_merged["confidence_score"],
    bins=[-0.01, 0.5, 0.8, 1.01],
    labels=["Low", "Medium", "High"],
)

print(accounts_merged["confidence"].value_counts())

confidence
High      178
Medium    101
Low         7
Name: count, dtype: int64


In [30]:
accounts_merged["flags"] = accounts_merged.apply(
    lambda r: "; ".join(FLAG_RULES[k] for k in FLAG_RULES if r[k]),
    axis=1,
)

print(accounts_merged.loc[accounts_merged["flags"] != "", ["account_name", "confidence", "flags"]].head(15).to_string(index=False))

                                             account_name confidence                                                                             flags
                                                  350.org     Medium                                                      ARR missing — imputed median
                             American Alliance of Museums       High                                Last contact date missing — neglect set to neutral
                       American Friends Service Committee       High                                                 Merged from duplicate CRM records
               American Foundation for Suicide Prevention       High                                Last contact date missing — neglect set to neutral
                                  The Akanksha Foundation     Medium                                                       No engagement signals found
                                  Aga Khan Foundation USA       High                          

In [31]:
EVENT_LABEL = {"demo_request": "demo request", "webinar": "webinar attendance",
               "content_download": "content download", "page_visit": "page visit",
               "email_open": "email open"}

top_ev = (events.sort_values("score", ascending=False)
                .groupby("matched_account").first()[["event_type", "event_date", "event_count"]])
arr_pct = accounts_merged["arr_num"].rank(pct=True)

def reason(r):
    bits = []
    p = arr_pct[r.name]
    if p >= 0.9:
        bits.append(f"top 10% by revenue (${r['arr_num']:,.0f})")
    elif p >= 0.7:
        bits.append(f"top 30% by revenue (${r['arr_num']:,.0f})")
    else:
        bits.append(f"${r['arr_num']:,.0f} ARR, {r['account_tier'] or 'untiered'}")

    if r["account_name"] in top_ev.index:
        t = top_ev.loc[r["account_name"]]
        d = (REF_DATE - t["event_date"]).days
        when = "this week" if d <= 7 else f"{d} days before data cutoff"
        bits.append(f"strongest signal: {EVENT_LABEL[t['event_type']]} ×{t['event_count']} {when}")
    else:
        bits.append("no engagement on record")

    if r["date_missing"]:
        bits.append("last contact unknown")
    else:
        ds = (REF_DATE - r["contact_date"]).days
        if ds >= 180:
            bits.append(f"no rep contact in {ds} days")
        elif ds >= 90:
            bits.append(f"last contacted {ds} days ago")
    return "; ".join(bits)

accounts_merged["why"] = accounts_merged.apply(reason, axis=1)

ranked = accounts_merged.sort_values("priority", ascending=False).reset_index(drop=True)
ranked.insert(0, "rank", ranked.index + 1)

for _, r in ranked.head(15).iterrows():
    print(f"{r['rank']:>3}. {r['account_name']}  [{r['priority']:.3f}]  confidence: {r['confidence']}")
    print(f"     why:   {r['why']}")
    if r["flags"]:
        print(f"     flags: {r['flags']}")
    print(f"     owner: {r['owner']}  |  fit {r['fit']:.2f}  intent {r['intent']:.2f}  neglect {r['neglect']:.2f}")
    print()

print("confidence mix:", ranked["confidence"].value_counts().to_dict())
print("low-confidence in top 20:", (ranked.head(20)["confidence"] == "Low").sum())

  1. Alley Cat Allies  [0.587]  confidence: High
     why:   top 30% by revenue ($70,450); strongest signal: content download ×9 this week
     owner: Rep C  |  fit 0.85  intent 0.90  neglect 0.19

  2. American Friends Service Committee  [0.581]  confidence: High
     why:   top 10% by revenue ($92,450); strongest signal: webinar attendance ×6 15 days before data cutoff; no rep contact in 236 days
     flags: Merged from duplicate CRM records
     owner: Rep A  |  fit 0.92  intent 0.48  neglect 1.00

  3. Education Development Center  [0.567]  confidence: High
     why:   top 10% by revenue ($120,000); strongest signal: webinar attendance ×5 8 days before data cutoff
     owner: Rep B  |  fit 0.94  intent 0.78  neglect 0.20

  4. No Kid Hungry  [0.525]  confidence: Medium
     why:   $26,850 ARR, Enterprise; strongest signal: content download ×4 14 days before data cutoff
     flags: ARR missing — imputed median
     owner: Rep C  |  fit 0.64  intent 1.00  neglect 0.39

  5. PATH  [0.

Download CSV

In [32]:
OUTPUT_COLS = ["rank", "account_name", "priority", "confidence", "why", "flags",
               "fit", "intent", "neglect", "arr_num", "account_tier", "industry",
               "region", "owner", "source_names", "source_arr"]

ranked[[c for c in OUTPUT_COLS if c in ranked.columns]].to_csv(
    "prioritized_accounts.csv", index=False)

review = ranked[(ranked["confidence"] == "Low") | ranked["arr_invalid"]
                | ranked["date_future"] | ranked["was_merged"]]
review[["rank", "account_name", "confidence", "flags",
        "source_names", "source_arr"]].to_csv("review_queue.csv", index=False)

print(f"Wrote prioritized_accounts.csv ({len(ranked)} rows) "
      f"and review_queue.csv ({len(review)} rows)")

Wrote prioritized_accounts.csv (286 rows) and review_queue.csv (27 rows)


In [33]:
try:
    from google.colab import files
    files.download("prioritized_accounts.csv")
    files.download("review_queue.csv")
except ImportError:
    print("Not running in Colab — CSVs written to the working directory.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Account Prioritization Agent — Requirements

## Interpretation

The VP asked for a ranked call list. The underlying problem is that reps choose
targets ad hoc, so high-value accounts showing active interest get the same
attention as accounts that will never convert. "Priority" is therefore defined as:
**an account worth winning that is showing interest now and is not already being
worked.** All three conditions matter — a large account with no activity is a
nurture, not a call.

The line "I'm not sure which fields are most useful" is read as delegating the
scoring design, not as a request to use every column.

## Scope and assumptions

In scope: ingest both exports, deduplicate, score, rank, and emit a prioritized
CSV plus a review queue, with a per-row explanation and confidence rating.

Assumptions:

- The book is mixed. 243 of 300 accounts have ARR above zero and 277 have a prior
  contact date, so these are mostly existing customers; 18 accounts have ARR of
  exactly 0.00 and 38 have none recorded. Accounts with revenue are treated as
  expansion targets, those without as prospects. There is no `stage` or `status`
  field to confirm this.
- Recency is measured against the latest date in each export (events end
  2026-07-28), not the wall clock, so results are reproducible.
- `industry` and `region` are excluded from scoring. With no conversion history,
  any industry weighting would encode assumption rather than evidence. Both are
  retained as filter columns.

## Approach

Priority = **fit × (0.7·intent + 0.3·neglect)**.

- **Fit** (static): 0.6 × ARR percentile + 0.4 × tier weight. Tier weights are set
  to each tier's median ARR percentile (Enterprise 0.89, Strategic 0.85,
  Mid-Market 0.58, SMB 0.22), so tier acts as a revenue proxy for the 40 accounts
  where ARR is unusable.
- **Intent** (behavioural): each event scored as
  `type_weight × log1p(count) × 0.5^(days_ago/30)`, summed per account and
  normalised. Log damping stops one inflated count dominating; exponential decay
  makes last week matter more than March. Email opens are weighted lowest —
  automated inbox scanning makes open counts unreliable as a human signal, visible
  here as a median of 22.5 opens per record against 2 for demo requests.
- **Neglect**: days since last rep contact, capped at 180 days.

Fit multiplies rather than adds, because both conditions must hold: a $120k
account with no activity scores near zero on intent and should not surface as a
call. A weighted average would let one strong factor mask a missing one.

**Deduplication** is on website domain, not name. 14 organisations appear twice
under an initialism and a full name (`AFSC` / `American Friends Service
Committee`); no string-similarity method connects those. Merged pairs carry
conflicting ARR and the export has no modification timestamp, so the merge takes
the maximum, retains both source values, and flags the record.

**Missing data is imputed neutral, never zero.** Zero-filling would penalise an
account for CRM hygiene rather than for anything real. ARR gaps take the median,
tier gaps take 0.4, an unknown contact date sets neglect to 0.5. Every imputation
is recorded as a flag with its reason, and the flags drive a confidence rating
(High 178 / Medium 101 / Low 7). Low-confidence records are ranked and shown, never
suppressed; records needing human correction — a string in the ARR column, a
negative value, four future-dated contacts — go to a separate review queue.

## Judging whether it works

The export has no outcome data, so ranking accuracy cannot be measured and no
accuracy figure is reported. Three things can be checked without it.

**Stability.** All weights, the five event-type weights, the fit blend, the
intent/neglect blend, and the decay half-life were perturbed ±20% across 200
runs. Spearman correlation against the baseline ranking held at 0.990 (worst run
0.967), and 83% of the top 20 was retained on average. The ordering is driven by
the data rather than by the chosen weights. The churn that does occur sits at the
boundary of the top 20, where scores are separated by hundredths of a point.

**Face validity.** Every row states its reason, so a rep can disagree specifically.
A disagreement points at a missing factor rather than at a bug.

**Coverage.** 178 of 286 accounts scored High confidence, 101 Medium, 7 Low, with
no Low-confidence account in the top 20, the head of the list is not built on
imputed values.

To really test it we would have to test it live like an A/B test. Half the team works the ranked list, half works as
today, compared on meetings booked per 100 calls over three to four weeks. That
also produces the first outcome labels, which is what would let the hand-set
weights be replaced with fitted ones.

## Non-goals and next steps

Not built: live CRM sync, scheduling, authentication, a UI, entity resolution
beyond domain and name normalisation. Learned weights are deliberately deferred —
supervised scoring needs conversion outcomes, which this export does not contain.
The scoring config is isolated so weights can be replaced with fitted coefficients
once that data exists.